# All prompt examples

In [ ]:
# ruff: noqa: T201, T203, E402
from __future__ import annotations

import logging
import os
from pathlib import Path

from dotenv import load_dotenv

while Path.cwd().name in ["notebooks", "research"]:
    os.chdir("..")

from src.formatting import format_gt_pairs_filepath, format_oracle_pairs_filepath, format_subsets_ontologies_paths
from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.prompts.prompt_utils import format_hierarchy
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)
from src.utils import read_oracle_pairs

logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

In [ ]:
DATASET_NAME, SET_NAME = "anatomy", "human-mouse"

src_onto_path, tgt_onto_path = format_subsets_ontologies_paths(DATASET_NAME, SET_NAME)

onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)

In [ ]:
src_entity, tgt_entity = read_oracle_pairs(format_oracle_pairs_filepath(DATASET_NAME, SET_NAME))[41] # consider testing: 34, 39, 41
try:
    src_entity, tgt_entity = OntologyEntryAttr(src_entity, onto_src), OntologyEntryAttr(tgt_entity, onto_tgt)
except AssertionError:
    src_entity, tgt_entity = OntologyEntryAttr(tgt_entity, onto_src), OntologyEntryAttr(src_entity, onto_tgt)

# New prompts instances

### Ontological prompts

In [ ]:
print(prompt_direct_entity_ontological(src_entity, tgt_entity))

In [ ]:
print(prompt_sequential_hierarchy_ontological(src_entity, tgt_entity))

### Natural language prompts

In [ ]:
print(prompt_direct_entity(src_entity, tgt_entity))

In [ ]:
print(prompt_sequential_hierarchy(src_entity, tgt_entity))

### Natural language prompts with synonyms

In [ ]:
print(prompt_direct_entity_with_synonyms(src_entity, tgt_entity))

In [ ]:
print(prompt_sequential_hierarchy_with_synonyms(src_entity, tgt_entity))

### Get full info

In [ ]:
print(format_hierarchy(src_entity.get_parents_by_levels(max_level=3), False, True))

In [ ]:
print(format_hierarchy(tgt_entity.get_parents_by_levels(max_level=3), False, True))

In [ ]:
print(src_entity)
print(src_entity.get_all_entity_names())
print(src_entity.get_parents_preferred_names())
print(src_entity.get_children_preferred_names())
print(src_entity.get_synonyms())
print(src_entity.get_direct_parents())
print(src_entity.get_direct_parents().pop().get_direct_children())

In [ ]:
print(tgt_entity)
print(tgt_entity.get_all_entity_names())
print(tgt_entity.get_parents_preferred_names())
print(tgt_entity.get_children_preferred_names())
print(tgt_entity.get_synonyms())
print(tgt_entity.get_direct_parents())
print(tgt_entity.get_direct_parents().pop().get_direct_children())

# Few-shot prompt examples

In [ ]:
GT_DATASET_NAME, GT_SET_NAME = "bioml-2024", "omim-ordo"
gt_src_onto_path, gt_tgt_onto_path = format_subsets_ontologies_paths(GT_DATASET_NAME, GT_SET_NAME)

gt_onto_src = OntologyAccess(gt_src_onto_path, annotate_on_init=True)
gt_onto_tgt = OntologyAccess(gt_tgt_onto_path, annotate_on_init=True)

In [ ]:
def get_gt_pairs(dataset_name: str, set_name: str) -> list[str]:
    with open(format_gt_pairs_filepath(dataset_name, set_name), "r") as f:
        gt_pairs = f.readlines()

    gt_pairs = [line.strip() for line in gt_pairs if line.strip()]
    gt_pairs = [(pair.split()[0], pair.split()[1]) for pair in gt_pairs]

    return gt_pairs


def parse_entity_pair(src: str, tgt: str, onto_src=None, onto_tgt=None) -> tuple[OntologyEntryAttr, OntologyEntryAttr]:
    """Try to parse source and target entities with their respective ontologies."""
    try:
        src_entity = OntologyEntryAttr(src, onto_src)
        tgt_entity = OntologyEntryAttr(tgt, onto_tgt)
    except AssertionError:
        src_entity = OntologyEntryAttr(tgt, onto_src)
        tgt_entity = OntologyEntryAttr(src, onto_tgt)

    return src_entity, tgt_entity


def make_few_shot_prompt(
    prompt_func,
    few_shot_pairs: list[tuple[str, str]],
    onto_src=None,
    onto_tgt=None,
    label: str = "True",
    chat_style: bool = True,
) -> str:
    prompt_str = ""

    for src, tgt in few_shot_pairs:
        src_entity, tgt_entity = parse_entity_pair(src, tgt, onto_src, onto_tgt)
        example_prompt = prompt_func(src_entity, tgt_entity)

        if chat_style:
            prompt_str += f"<|user|>\n{example_prompt.strip()}\n<|assistant|>\n{label}\n\n"
        else:
            example_prompt = example_prompt.replace('Respond with "True" or "False"', label)
            prompt_str += example_prompt.strip() + f"\n{label}\n\n"

    return prompt_str


def make_full_prompt(*few_shot_parts: str, prompt_str: str, chat_style: bool = True) -> str:
    few_shot_str = "".join(part for part in few_shot_parts if part.strip())

    if chat_style:
        return f"{few_shot_str}<|user|>\n{prompt_str.strip()}\n<|assistant|>\n"
    else:
        return few_shot_str + prompt_str.strip()

In [ ]:
gt_pairs = get_gt_pairs(GT_DATASET_NAME, GT_SET_NAME)[5:6]
not_gt_pairs = get_gt_pairs(GT_DATASET_NAME, GT_SET_NAME)[7:9]
not_gt_pairs = [
    (not_gt_pairs[0][0], not_gt_pairs[1][1]),
    (not_gt_pairs[1][0], not_gt_pairs[0][1])
][:1]

few_shot_true = make_few_shot_prompt(
    prompt_func=prompt_sequential_hierarchy, 
    few_shot_pairs=gt_pairs, 
    onto_src=gt_onto_src, 
    onto_tgt=gt_onto_tgt,
    label="True",
    chat_style=True
)

few_shot_false = make_few_shot_prompt(
    prompt_func=prompt_sequential_hierarchy, 
    few_shot_pairs=not_gt_pairs, 
    onto_src=gt_onto_src, 
    onto_tgt=gt_onto_tgt,
    label="False",
    chat_style=True
)

prompt = prompt_sequential_hierarchy(src_entity, tgt_entity)

full_prompt = make_full_prompt(
    few_shot_true,
    few_shot_false,
    prompt_str=prompt,
    chat_style=True
)

In [ ]:
print(full_prompt)